----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Filings - Retrieval and Downloads
This standalone notebook demonstrates `lseg.data.content.filings.retrieval`: retrieve file metadata by identifier, download one file or a collection, and use asynchronous downloads with a callback. It also shows how to download directly from a search response.

**Prerequisites:** a configured platform session entitled to Filings retrieval and search. The identifiers below come from the tests; their availability depends on the service and your entitlements. See [Search](EX-2.12.01-Filings-Search.ipynb) to discover other identifiers.

Retrieval obtains signed download URLs; downloading is a separate action. Signed URLs are temporary access links: do not share them or save them in notebook outputs. Examples display metadata without the `SignedURL` column. No document is uploaded or modified on the service.

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. Session settings and credentials are read from `lseg-data.config.json` in the project's `Configuration` directory.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import lseg.data as ld
from lseg.data.content import filings
from lseg.data.content.filings import Feed

## Open the data session
Select the `platform.rdp` session configured in `lseg-data.config.json`. Change this name if your entitled platform session has a different name.

In [ ]:
ld.open_session("platform.rdp")

## Retrieve file metadata by identifier
A retrieval `Definition` requires **exactly one** of these parameters:

| Parameter | Identifier | Test example |
|---|---|---|
| `filename` | Service filename, including its format identifier | `ecpfilings_34359955599_pdf` |
| `dcn` | Document Control Number | `cr00329072` |
| `doc_id` | LSEG document ID | `49612437` |
| `filing_id` | LSEG Financial Filing ID | `34359955599` |

The example uses `doc_id`, which can return several file formats. To try another identifier, replace the definition below, for example with `filings.retrieval.Definition(dcn="cr00329072")`. These examples need not all refer to the same filing.

`response.data.df` contains `Filename`, `SignedURL` and `MimeType`. `response.data.files` provides the corresponding downloadable file objects. A filename-based retrieval may not supply a MIME type.

In [ ]:
response = filings.retrieval.Definition(doc_id="49612437").get_data()
response.data.df.drop(columns=["SignedURL"])

### Validate the identifier rule locally
The test checks that combining identifiers raises `ValueError`. Validation occurs when constructing the definition; this cell does not make a service request.

In [ ]:
try:
    filings.retrieval.Definition(doc_id="49612437", filename="file")
except ValueError as error:
    print(f"Expected validation error: {error}")
else:
    raise RuntimeError("A retrieval definition must accept only one identifier")

## Prepare download destinations and check results
The download API expects an existing directory and can replace an existing file with the same name. This notebook creates a unique directory under the operating system's temporary directory, then a separate subdirectory for each example to avoid overwriting earlier downloads. The printed location lets you inspect the files afterwards; they are not deleted automatically.

Some download operations return errors in the response instead of raising them, especially collection and asynchronous downloads. The helper below checks both `errors` and each file's `is_success` before reporting success, and verifies that downloaded files exist.

In [ ]:
download_root = Path(mkdtemp(prefix="lseg-filings-"))
print(f"Download directory: {download_root}")


def check_download(result):
    if result.errors:
        raise RuntimeError(f"Filings download failed: {result.errors}")
    if not result.data.files:
        raise RuntimeError("Filings download returned no files")
    for downloaded in result.data.files:
        if not downloaded.is_success or not downloaded.file_location:
            raise RuntimeError("A filing file was not downloaded successfully")
        location = Path(downloaded.file_location)
        if not location.is_file():
            raise FileNotFoundError(location)
        print(location)

## Download one file synchronously
`files[0].download(path=...)` downloads the first available format. The library determines the file extension from the response; use `file_location` rather than constructing a local filename yourself. An empty retrieval result is reported explicitly before indexing.

In [ ]:
files = response.data.files
if not files:
    raise RuntimeError("No downloadable files were returned for this document")

single_directory = download_root / "single"
single_directory.mkdir()
download_response = files[0].download(path=str(single_directory))
check_download(download_response)

## Download all returned formats synchronously
`files.download(path=...)` downloads the collection. A document may have HTML, PDF or text variants. A collection download can partially succeed, so check its returned errors rather than assuming that every file was downloaded.

In [ ]:
all_directory = download_root / "all-formats"
all_directory.mkdir()
all_downloads = files.download(path=str(all_directory))
check_download(all_downloads)

## Retrieve and download asynchronously
Use `get_data_async()` for retrieval and `download_async()` for downloading. Jupyter supports top-level `await`. A collection callback receives the completed download response; it is not a per-byte progress callback. Checking the awaited result explicitly ensures failures are surfaced even if a callback is not invoked.

Retrieve metadata again to obtain fresh signed URLs if earlier links have expired.

In [ ]:
def on_download_complete(result):
    print(f"Download callback: {len(result.data.files)} file results received")


async_response = await filings.retrieval.Definition(doc_id="49612437").get_data_async()
display(async_response.data.df.drop(columns=["SignedURL"]))

async_directory = download_root / "async-all-formats"
async_directory.mkdir()
async_downloads = await async_response.data.files.download_async(
    path=str(async_directory),
    callback=on_download_complete,
)
check_download(async_downloads)

## Download directly from search results
Search file objects also expose `download()` and `download_async()`. They resolve the identifiers into retrieval URLs for you. This example searches for one filing, then downloads its first available file asynchronously. `limit=1` limits filing hits, not necessarily the number of file formats. To download the entire search collection instead, call `search_response.data.files.download_async(path=...)` and check the returned result.

In [ ]:
search_response = await filings.search.Definition(
    form_type="10-Q",
    feed=Feed.EDGAR,
    start_date="2020-10-01",
    end_date="2020-12-31",
    limit=1,
).get_data_async()
display(search_response.data.df)

search_files = search_response.data.files
if not search_files:
    raise RuntimeError("No downloadable files matched the filing search")

search_directory = download_root / "from-search"
search_directory.mkdir()
search_download = await search_files[0].download_async(
    path=str(search_directory),
    callback=on_download_complete,
)
check_download(search_download)

## Close the session
If an earlier request or download fails, run this cell to close the session. Downloads remain in the printed temporary directory for inspection; remove that specific directory manually when it is no longer needed. Clear notebook outputs before sharing, especially if you displayed signed URLs or raw retrieval responses.

In [ ]:
ld.close_session()